# Benchmarking

In [1]:
from src.data import load_data, split_by_year
from src.benchmark import zpp_components, fit_caps, altman_zpp, zpp_zone, ZScoreToPD
from src.metrics import evaluate
import pandas as pd

In [2]:
train, val, test = split_by_year(load_data())

In [3]:
caps = fit_caps(zpp_components(train))
z_tr, z_va, z_te = altman_zpp(train, caps), altman_zpp(val, caps), altman_zpp(test, caps)

In [4]:
pd_map = ZScoreToPD().fit(z_tr, train["default"])
p_tr, p_va, p_te = pd_map.predict_proba(z_tr), pd_map.predict_proba(z_va), pd_map.predict_proba(z_te)

In [5]:
pd.DataFrame({'train': evaluate(train["default"], p_tr),
              "val": evaluate(val["default"], p_va),
              "test": evaluate(test["default"], p_te)
              }).round(4)

,train,val,test
n,55927.0000,10473.0000,12282.0000
n_pos,403.0000,87.0000,119.0000
base_rate,0.0072,0.0083,0.0097
pr_auc,0.0180,0.0198,0.0230
pr_auc_lift,2.4921,2.3862,2.3697
roc_auc,0.7634,0.7878,0.7731
brier,0.0071,0.0082,0.0096
brier_skill,0.0008,-0.0002,-0.0001
mean_pred,0.0072,0.0076,0.0076
ks,0.4562,0.5815,0.5260


Calibration-in-the-large drifts out of time. The Z → PD map is fitted on train (0.72% base rate) and predicts a mean PD of 0.76% on both val and test, against observed rates of 0.83% and 0.97%, so the test PDs come out about 22% too low. This is left uncorrected: the benchmark is meant to stay fixed, and any model fitted on train will face the same shift in the base rate.

The published Z'' zones (distress < 1.1, grey 1.1 - 2.6, safe > 2.6) on the test window:

In [6]:
zones = (test.assign(zone=zpp_zone(z_te))
         .groupby("zone", observed=True)["default"]
         .agg(firm_years="size", defaults="sum", rate="mean"))
zones["lift"] = zones["rate"] / test["default"].mean()
zones.round(4)

,firm_years,defaults,rate,lift
zone,,,,
distress,5058,105,0.0208,2.1426
grey,1942,5,0.0026,0.2657
safe,5282,9,0.0017,0.1759


In [7]:
distress = zones.loc["distress"]
print(f"{distress.defaults / zones.defaults.sum():.1%} of test defaults fall in the distress zone, "
      f"{distress.firm_years / zones.firm_years.sum():.1%} of the book")

88.2% of test defaults fall in the distress zone, 41.2% of the book
